# 05_00 - Phase 5 Run Order and Input Validation

This notebook validates Phase 5 inputs and records the run order. It does not train models.


## Run Order

1. `05_01_LightGBM_Raw.ipynb`
2. `05_02_XGBoost_Raw.ipynb`
3. `05_03_MLP_Raw.ipynb`
4. `05_04_CNN_BiLSTM_Sequence.ipynb`
5. `05_05_Weighted_Blending.ipynb`
6. `05_06_Stacking_Calibration.ipynb`
7. `05_Hybrid_Ensemble.ipynb`


In [1]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except ImportError:
except ImportError:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 tr...: ném lỗi và dừng cell
    raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 training locally.")


In [2]:
# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount("/content/drive"): mount Google Drive trên Colab
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# import gc: giải phóng bộ nhớ
import gc
# import importlib: import thư viện importlib
import importlib
# import json: đọc/ghi JSON metadata
import json
# import os: biến môi trường hệ thống
import os
# import random: cố định seed ngẫu nhiên
import random
# import subprocess: chạy lệnh pip/cài package
import subprocess
# import sys: tham số Python runtime
import sys
# import time: đo thời gian thực thi
import time
# from datetime import datetime, timezone: import thư viện datetime
from datetime import datetime, timezone
# from itertools import combinations, product: import thư viện itertools
from itertools import combinations, product
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import joblib: lưu/tải object đã fit
import joblib
# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# from sklearn.metrics import (: thư viện machine learning scikit-learn
from sklearn.metrics import (
    # accuracy_score,: thực thi lệnh Python
    accuracy_score,
    # average_precision_score,: thực thi lệnh Python
    average_precision_score,
    # brier_score_loss,: thực thi lệnh Python
    brier_score_loss,
    # confusion_matrix,: thực thi lệnh Python
    confusion_matrix,
    # f1_score,: thực thi lệnh Python
    f1_score,
    # precision_recall_fscore_support,: thực thi lệnh Python
    precision_recall_fscore_support,
    # roc_auc_score,: thực thi lệnh Python
    roc_auc_score,
# ): đóng ngoặc gọi hàm
)

# SEED: biến cấu hình/hằng số của notebook
SEED = 42
# FAKE_LABEL: biến cấu hình/hằng số của notebook
FAKE_LABEL = 1
# REAL_LABEL: biến cấu hình/hằng số của notebook
REAL_LABEL = 0
# DEFAULT_THRESHOLD: biến cấu hình/hằng số của notebook
DEFAULT_THRESHOLD = 0.50
# TARGET_PRECISION_FAKE: biến cấu hình/hằng số của notebook
TARGET_PRECISION_FAKE = 0.975

# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = Path(os.environ.get("FAKE_REVIEWS_PROJECT_ROOT", "/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews"))
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / "artifacts" / "features"
# MODEL_DIR: biến cấu hình/hằng số của notebook
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
# ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
# PREDICTION_DIR: biến cấu hình/hằng số của notebook
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
# REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
# REPORT_FIGURE_DIR: biến cấu hình/hằng số của notebook
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
# PROCESSED_DIR: biến cấu hình/hằng số của notebook
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# for: vòng lặp — for directory in [MODEL_DIR, ENSEMBLE_DIR, PREDICTION_DIR, R
for directory in [MODEL_DIR, ENSEMBLE_DIR, PREDICTION_DIR, REPORT_TABLE_DIR, REPORT_FIGURE_DIR]:
    # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    directory.mkdir(parents=True, exist_ok=True)

# RAW_FEATURE_PATHS: biến cấu hình/hằng số của notebook
RAW_FEATURE_PATHS = {
    # "train": FEATURE_DIR / "features_raw_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "features_raw_train.npy",
    # "val": FEATURE_DIR / "features_raw_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "features_raw_val.npy",
    # "test": FEATURE_DIR / "features_raw_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "features_raw_test.npy",
# }: đóng khối từ điển
}
# LABEL_PATHS: biến cấu hình/hằng số của notebook
LABEL_PATHS = {
    # "train": FEATURE_DIR / "labels_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "labels_train.npy",
    # "val": FEATURE_DIR / "labels_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "labels_val.npy",
    # "test": FEATURE_DIR / "labels_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "labels_test.npy",
# }: đóng khối từ điển
}
# FEATURE_METADATA_PATH: biến cấu hình/hằng số của notebook
FEATURE_METADATA_PATH = FEATURE_DIR / "feature_metadata.json"

# random.seed(SEED): cố định seed random
random.seed(SEED)
# np.random.seed(SEED): cố định seed numpy
np.random.seed(SEED)


# utc_now: định nghĩa hàm utc now
def utc_now() -> str:
    # return: trả kết quả từ hàm
    return datetime.now(timezone.utc).isoformat()


# ensure_package: import hoặc pip install package
def ensure_package(import_name: str, pip_name: str | None = None):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)
    # except: xử lý ngoại lệ — except ImportError:
    except ImportError:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or impor...: thực thi lệnh Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)


# read_json: đọc file JSON
def read_json(path: Path, default=None):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # return: trả kết quả từ hàm
        return default if default is not None else {}
    # with: context manager — with path.open("r", encoding="utf-8") as file:
    with path.open("r", encoding="utf-8") as file:
        # return: parse nội dung JSON
        return json.load(file)


# load_raw_arrays: nạp feature/label .npy từ Phase 2
def load_raw_arrays():
    # missing = ...: kiểm tra file/thư mục tồn tại
    missing = [str(path) for path in list(RAW_FEATURE_PATHS.values()) + list(LABEL_PATHS.values()) if not path.exists()]
    # if: điều kiện — if missing:
    if missing:
        # raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(miss...: ghi dictionary ra JSON
        raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(missing, indent=2))
    # X: biến cấu hình/hằng số của notebook
    X = {split: np.load(path).astype(np.float32, copy=False) for split, path in RAW_FEATURE_PATHS.items()}
    # y = ...: nạp mảng từ file .npy
    y = {split: np.load(path).astype(np.int64, copy=False) for split, path in LABEL_PATHS.items()}
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # if: điều kiện — if X[split].ndim != 2:
        if X[split].ndim != 2:
            # raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}"): ném lỗi và dừng cell
            raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}")
        # if: điều kiện — if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
        if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
            # raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[spl...: ném lỗi và dừng cell
            raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[split].shape}")
        # if: điều kiện — if not np.isfinite(X[split]).all():
        if not np.isfinite(X[split]).all():
            # raise ValueError(f"Non-finite raw features in {split}"): ném lỗi và dừng cell
            raise ValueError(f"Non-finite raw features in {split}")
    # return: trả kết quả từ hàm
    return X, y


# safe_roc_auc: ROC-AUC an toàn
def safe_roc_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(roc_auc_score(y_true, prob_fake))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# safe_pr_auc: PR-AUC an toàn
def safe_pr_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(average_precision_score(y_true, prob_fake, pos_label=FAKE_LABEL))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# evaluate_predictions: tính metric classification từ xác suất
def evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=DEFAULT_THRESHOLD, threshold_strategy="default_0.5"):
    # y_true = ...: ép kiểu dữ liệu cột
    y_true = np.asarray(y_true).astype(int)
    # prob_fake = ...: ép kiểu dữ liệu cột
    prob_fake = np.asarray(prob_fake).astype(float)
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob_fake >= threshold).astype(int)
    # precision, recall, f1, support = precision_recall_fscore_support(: thực thi lệnh Python
    precision, recall, f1, support = precision_recall_fscore_support(
        # y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0: thực thi lệnh Python
        y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0
    # ): đóng ngoặc gọi hàm
    )
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL...: thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL]).ravel()
    # return: trả kết quả từ hàm
    return {
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "seed": int(SEED),: ép kiểu số nguyên
        "seed": int(SEED),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "model_variant": model_variant,: thực thi lệnh Python
        "model_variant": model_variant,
        # "threshold": float(threshold),: ép kiểu số thực
        "threshold": float(threshold),
        # "threshold_strategy": threshold_strategy,: ép kiểu chuỗi
        "threshold_strategy": threshold_strategy,
        # "accuracy": float(accuracy_score(y_true, y_pred)),: ép kiểu số thực
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),: ép kiểu số thực
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # "precision_fake": float(precision[1]),: ép kiểu số thực
        "precision_fake": float(precision[1]),
        # "recall_fake": float(recall[1]),: ép kiểu số thực
        "recall_fake": float(recall[1]),
        # "f1_fake": float(f1[1]),: ép kiểu số thực
        "f1_fake": float(f1[1]),
        # "support_real": int(support[0]),: ép kiểu số nguyên
        "support_real": int(support[0]),
        # "support_fake": int(support[1]),: ép kiểu số nguyên
        "support_fake": int(support[1]),
        # "roc_auc": safe_roc_auc(y_true, prob_fake),: thực thi lệnh Python
        "roc_auc": safe_roc_auc(y_true, prob_fake),
        # "pr_auc": safe_pr_auc(y_true, prob_fake),: thực thi lệnh Python
        "pr_auc": safe_pr_auc(y_true, prob_fake),
        # "brier_score": float(brier_score_loss(y_true, prob_fake)),: ép kiểu số thực
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        # "tn": int(tn),: ép kiểu số nguyên
        "tn": int(tn),
        # "fp": int(fp),: ép kiểu số nguyên
        "fp": int(fp),
        # "fn": int(fn),: ép kiểu số nguyên
        "fn": int(fn),
        # "tp": int(tp),: ép kiểu số nguyên
        "tp": int(tp),
    # }: đóng khối từ điển
    }


# save_probability: lưu xác suất fake ra file .npy
def save_probability(prob_fake, model_variant, split):
    # path = ...: gán giá trị cho biến path
    path = PREDICTION_DIR / f"{model_variant}_{split}_prob.npy"
    # np.save(path, np.asarray(prob_fake, dtype=np.float32)): lưu mảng numpy ra file .npy
    np.save(path, np.asarray(prob_fake, dtype=np.float32))
    # return: trả kết quả từ hàm
    return str(path)


# write_metrics: ghi bảng metric ra CSV
def write_metrics(prob_map, y, model_variant, output_name):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split=split, model_variant=model_variant)
        # row["probability_path"] = save_probability(prob_map[split], model_variant, split...: thực thi lệnh Python
        row["probability_path"] = save_probability(prob_map[split], model_variant, split)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # path = ...: gán giá trị cho biến path
    path = REPORT_TABLE_DIR / output_name
    # df.to_csv(path, index=False): ghi DataFrame ra file CSV
    df.to_csv(path, index=False)
    # display(df): hiển thị bảng/kết quả trên notebook
    display(df)
    # print("Saved metrics:", path): in thông tin ra console
    print("Saved metrics:", path)
    # return: trả kết quả từ hàm
    return df, path


In [4]:
# feature_metadata = ...: đọc file JSON
feature_metadata = read_json(FEATURE_METADATA_PATH)
# validation_rows = ...: gán giá trị cho biến validation rows
validation_rows = []
# for: vòng lặp — for split in ["train", "val", "test"]:
for split in ["train", "val", "test"]:
    # X_probe = ...: nạp mảng từ file .npy
    X_probe = np.load(RAW_FEATURE_PATHS[split], mmap_mode="r")
    # y_probe = ...: nạp mảng từ file .npy
    y_probe = np.load(LABEL_PATHS[split], mmap_mode="r")
    # validation_rows.append({: thực thi lệnh Python
    validation_rows.append({
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "feature_path": str(RAW_FEATURE_PATHS[split]),: ép kiểu chuỗi
        "feature_path": str(RAW_FEATURE_PATHS[split]),
        # "label_path": str(LABEL_PATHS[split]),: ép kiểu chuỗi
        "label_path": str(LABEL_PATHS[split]),
        # "rows": int(X_probe.shape[0]),: ép kiểu số nguyên
        "rows": int(X_probe.shape[0]),
        # "feature_dim": int(X_probe.shape[1]),: ép kiểu số nguyên
        "feature_dim": int(X_probe.shape[1]),
        # "label_rows": int(y_probe.shape[0]),: ép kiểu số nguyên
        "label_rows": int(y_probe.shape[0]),
        # "status": "pass" if X_probe.shape[0] == y_probe.shape[0] else "fail",: thực thi lệnh Python
        "status": "pass" if X_probe.shape[0] == y_probe.shape[0] else "fail",
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # del X_probe, y_probe: xóa biến để giải phóng RAM/VRAM
    del X_probe, y_probe

# run_order_df = ...: gán giá trị cho biến run order df
run_order_df = pd.DataFrame([
    # {"order": 1, "notebook": "05_01_LightGBM_Raw.ipynb", "output_prefix": "phase5_lg...: thực thi lệnh Python
    {"order": 1, "notebook": "05_01_LightGBM_Raw.ipynb", "output_prefix": "phase5_lgbm_raw"},
    # {"order": 2, "notebook": "05_02_XGBoost_Raw.ipynb", "output_prefix": "phase5_xgb...: thực thi lệnh Python
    {"order": 2, "notebook": "05_02_XGBoost_Raw.ipynb", "output_prefix": "phase5_xgb_raw"},
    # {"order": 3, "notebook": "05_03_MLP_Raw.ipynb", "output_prefix": "phase5_mlp_raw...: thực thi lệnh Python
    {"order": 3, "notebook": "05_03_MLP_Raw.ipynb", "output_prefix": "phase5_mlp_raw"},
    # {"order": 4, "notebook": "05_04_CNN_BiLSTM_Sequence.ipynb", "output_prefix": "ph...: thực thi lệnh Python
    {"order": 4, "notebook": "05_04_CNN_BiLSTM_Sequence.ipynb", "output_prefix": "phase5_cnn_bilstm_sequence"},
    # {"order": 5, "notebook": "05_05_Weighted_Blending.ipynb", "output_prefix": "phas...: đếm số phần tử
    {"order": 5, "notebook": "05_05_Weighted_Blending.ipynb", "output_prefix": "phase5_weighted_blend"},
    # {"order": 6, "notebook": "05_06_Stacking_Calibration.ipynb", "output_prefix": "p...: thực thi lệnh Python
    {"order": 6, "notebook": "05_06_Stacking_Calibration.ipynb", "output_prefix": "phase5_stacking_calibrated"},
    # {"order": 7, "notebook": "05_Hybrid_Ensemble.ipynb", "output_prefix": "phase5_fi...: thực thi lệnh Python
    {"order": 7, "notebook": "05_Hybrid_Ensemble.ipynb", "output_prefix": "phase5_final_selection"},
# ]): đóng list comprehension hoặc danh sách
])
# validation_df = ...: gán giá trị cho biến validation df
validation_df = pd.DataFrame(validation_rows)
# validation_df.to_csv(REPORT_TABLE_DIR / "phase5_run_order_input_validation.csv",...: ghi DataFrame ra file CSV
validation_df.to_csv(REPORT_TABLE_DIR / "phase5_run_order_input_validation.csv", index=False)
# run_order_df.to_csv(REPORT_TABLE_DIR / "phase5_run_order.csv", index=False): ghi DataFrame ra file CSV
run_order_df.to_csv(REPORT_TABLE_DIR / "phase5_run_order.csv", index=False)
# display(validation_df): hiển thị bảng/kết quả trên notebook
display(validation_df)
# display(run_order_df): hiển thị bảng/kết quả trên notebook
display(run_order_df)
# print("ModernBERT model:", feature_metadata.get("bert", {}).get("model_name")): in thông tin ra console
print("ModernBERT model:", feature_metadata.get("bert", {}).get("model_name"))


,generated_at_utc,split,feature_path,label_path,rows,feature_dim,label_rows,status
0,2026-06-09T17:00:26.850229+00:00,train,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...,29923,777,29923,pass
1,2026-06-09T17:00:27.572000+00:00,val,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...,6413,777,6413,pass
2,2026-06-09T17:00:28.401581+00:00,test,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...,6413,777,6413,pass


,order,notebook,output_prefix
0,1,05_01_LightGBM_Raw.ipynb,phase5_lgbm_raw
1,2,05_02_XGBoost_Raw.ipynb,phase5_xgb_raw
2,3,05_03_MLP_Raw.ipynb,phase5_mlp_raw
3,4,05_04_CNN_BiLSTM_Sequence.ipynb,phase5_cnn_bilstm_sequence
4,5,05_05_Weighted_Blending.ipynb,phase5_weighted_blend
5,6,05_06_Stacking_Calibration.ipynb,phase5_stacking_calibrated
6,7,05_Hybrid_Ensemble.ipynb,phase5_final_selection


ModernBERT model: answerdotai/ModernBERT-base
